[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MLManuals/Deep-Learning/blob/main/chapter18_sequence_models_and_embeddings/notebook.ipynb)

# Chapter 18: Sequence Models and Embeddings

Run the cell below once, first, to install any missing packages (only needed on Google Colab; skip it if you're running locally and already have them).

In [ ]:
%pip install -q torch numpy scikit-learn

Convolution solved a problem of space: the same feature can appear anywhere in a picture. Language, speech, music and prices raise a problem of time. The meaning of a word depends on the words before it, sometimes many words before it, and the order is the information. *The dog bit the man* and *the man bit the dog* contain exactly the same words.

In 1990 Jeffrey Elman, a cognitive scientist at the University of California, San Diego, published a paper called *Finding Structure in Time*. His network was small and his change to it was simpler still. As it read a sequence one item at a time, he copied its hidden layer from the previous step back in as extra input, so that the network always had a summary of what it had seen so far. Elman tested it on tasks ranging from a version of Chapter 3's XOR spread out over time to sentences, and found that, with no grammar given to it, the network's hidden states sorted words into nouns and verbs by itself. The recurrent connections, he wrote, are what give the network memory.

This chapter is a first look, built on sequences you generate yourself. Book 8 takes the ideas to real language. Before you read on, try to guess the answers to these four questions.

1. **If a model only counts which symbols appear, what can it never learn?**
2. **How do you feed a word into a network that only does arithmetic?**
3. **How can a network with a fixed number of weights read a sequence of any length?**
4. **Why would remembering something from fifty steps ago be hard?**

## Data That Comes in Order

To prove that order matters you need a task where order is the *only* clue. Here is one. Each sequence is thirty symbols long. Symbols 3 to 9 are filler. Each sequence also contains exactly one 1 and exactly one 2, at random positions. The label is 1 if the 1 comes first and 0 if the 2 comes first.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

The generator below builds sequences of filler and then drops a 1 and a 2 into two random positions, in the order the label calls for.

In [ ]:
def make_order(n, length, seed):
    g = torch.Generator().manual_seed(seed)
    X = torch.randint(3, 10, (n, length), generator=g)
    y = torch.randint(0, 2, (n,), generator=g)
    for i in range(n):
        spots = torch.randperm(length, generator=g)[:2].sort().values
        first, second = (1, 2) if y[i] == 1 else (2, 1)
        X[i, spots[0]], X[i, spots[1]] = first, second
    return X, y

X, y = make_order(3, 12, seed=5)
for row, label in zip(X.tolist(), y.tolist()):
    print(row, label)

Find the 1 and the 2 in each row and you can check the labels yourself. Now notice what a model that ignores order would see. Every sequence contains one 1, one 2 and some filler, whichever label it has. If you only count which symbols are present, the two classes are identical, and no amount of training can do better than a coin toss. Every network in this book until now would be in that position if it was given the symbols with their order thrown away.

## Embeddings, Turning Symbols into Vectors

Before order, a smaller problem. The symbols here are numbers, but they are names, not quantities. Symbol 8 is not twice symbol 4, any more than the word *dog* is twice the word *cat*. Feeding them in as numbers would invent a meaning that is not there.

The fix is an **embedding**: a table with one row of learned numbers per symbol. To feed symbol 4 to a network, you look up row 4 and use that vector. The rows start random and are trained by backpropagation along with every other weight, so the network decides for itself what each symbol should look like.

> 🖼️ **Figure 18.1:** An embedding is a lookup. Each symbol has its own row of learned numbers, and feeding a symbol to the network means handing it that row.

In [ ]:
torch.manual_seed(0)
emb = nn.Embedding(10, 16)
print(emb.weight.shape)
looked_up = emb(torch.tensor([4]))
one_hot = torch.zeros(1, 10)
one_hot[0, 4] = 1.0
print(torch.equal(looked_up, one_hot @ emb.weight))

The second line is this book's habit of checking a new tool against something already understood. Looking up row 4 gives exactly the same vector as multiplying a one hot row, all zeros except a 1 in position 4, by the table. So an embedding is just a linear layer whose input is one hot, implemented as a lookup because that is far cheaper than multiplying by a long row of zeros. With a real vocabulary of fifty thousand words that saving is enormous.

Elman's network sorting nouns from verbs was an early sign of what embeddings would become. Trained on enough text, the rows for words used in similar ways end up close together, and in 2013 a team at Google showed that the differences between rows could even capture relationships, so that the vector from *man* to *woman* was close to the vector from *king* to *queen*. Book 8 starts there.

## The Recurrent Idea

A **recurrent neural network** reads one symbol per step. It keeps a vector called the **hidden state**, and at every step it combines the new input with the previous hidden state to make the next one. The same weights are used at every step, which is how a fixed set of weights reads a sequence of any length, just as Chapter 17's filters were reused at every position.

> 🖼️ **Figure 18.2:** A recurrent network unrolled over four steps. It is one cell with one set of weights, drawn once per step. The hidden state carries a summary of everything read so far, and the answer is read off the last one.

The update is short enough to write by hand: $h_t = \tanh(W_{x} x_t + W_{h} h_{t-1} + b)$. Here it is for five steps of four random inputs, beside PyTorch's `nn.RNN`, which should agree exactly.

In [ ]:
torch.manual_seed(0)
rnn = nn.RNN(input_size=4, hidden_size=3, batch_first=True)
inputs = torch.randn(1, 5, 4)

h = torch.zeros(3)
for t in range(5):
    h = torch.tanh(rnn.weight_ih_l0 @ inputs[0, t] + rnn.bias_ih_l0
                   + rnn.weight_hh_l0 @ h + rnn.bias_hh_l0)
outputs, last = rnn(inputs)
print([round(v, 4) for v in h.tolist()])
print([round(v, 4) for v in last[0, 0].tolist()])
print(torch.allclose(h, last[0, 0]))

The loop you wrote is the whole recurrent idea, and PyTorch's version keeps two bias vectors where the formula has one, which changes nothing since they are simply added together. Now put an embedding in front and a dense layer behind, and compare it with a model that averages the embeddings, which keeps the symbols but throws their order away.

In [ ]:
class Bag(nn.Module):
    def __init__(self, vocab=10, dim=16):
        super().__init__()
        self.emb = nn.Embedding(vocab, dim)
        self.out = nn.Linear(dim, 2)
    def forward(self, x):
        return self.out(self.emb(x).mean(1))

class Reader(nn.Module):
    def __init__(self, cell, vocab=10, dim=16, hidden=32):
        super().__init__()
        self.emb = nn.Embedding(vocab, dim)
        self.rnn = cell(dim, hidden, batch_first=True)
        self.out = nn.Linear(hidden, 2)
    def forward(self, x):
        states, _ = self.rnn(self.emb(x))
        return self.out(states[:, -1])

`Reader` takes the recurrent cell as an argument, so the same class will serve for a plain RNN now and an LSTM later. The training function is Chapter 12's with a test at the end.

In [ ]:
loss_fn = nn.CrossEntropyLoss()

def run(model, maker, length, epochs=20, seed=0):
    Xtr, ytr = maker(2000, length, 1)
    Xte, yte = maker(500, length, 2)
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=0.01)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=64, shuffle=True,
                        generator=torch.Generator().manual_seed(seed))
    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            opt.zero_grad()
            loss_fn(model(xb), yb).backward()
            opt.step()
    model.eval()
    with torch.no_grad():
        return round((model(Xte).argmax(1) == yte).float().mean().item(), 3)

It takes the data generator as an argument too, so the same function will run both tasks in this chapter. Now train the two models on sequences of thirty symbols.

In [ ]:
torch.manual_seed(0)
bag = Bag()
torch.manual_seed(0)
reader = Reader(nn.RNN)
print("bag", run(bag, make_order, 30))
print("rnn", run(reader, make_order, 30))

The order blind model scores what the argument predicted, a coin toss. The recurrent network gets every one of the 500 test sequences right. Same embeddings, same data, same training; the only difference is that one of them reads in order.

### The Trouble with Long Memories

The recurrent network has a weakness, and you have met it twice. Training it means backpropagating through every step, and the unrolled network in Figure 18.2 is just a very deep network whose depth is the length of the sequence. Chapter 4's whisper line and Chapter 16's twenty layers apply again, this time across time. Measure how much gradient reaches each step's input on sequences of length 100, at the start of training.

In [ ]:
def make_first(n, length, seed):
    g = torch.Generator().manual_seed(seed)
    X = torch.randint(3, 10, (n, length), generator=g)
    y = torch.randint(0, 2, (n,), generator=g)
    X[:, 0] = torch.where(y == 1, 1, 2)
    return X, y

This second generator puts the only useful symbol, a 1 or a 2, at the very start, and fills the rest with noise. To answer, a model has to carry that first symbol all the way to the end.

In [ ]:
X, y = make_first(64, 100, seed=3)
for cell in [nn.RNN, nn.LSTM]:
    torch.manual_seed(0)
    model = Reader(cell)
    e = model.emb(X).detach().requires_grad_(True)
    states, _ = model.rnn(e)
    loss_fn(model.out(states[:, -1]), y).backward()
    size = e.grad.norm(dim=(0, 2))
    print(cell.__name__, ["%.0e" % size[99 - back].item()
                          for back in [0, 10, 25, 50, 99]])

The five columns are the gradient reaching the input zero, 10, 25, 50 and 99 steps back from the end. For the plain RNN it has shrunk about ten million times by 25 steps back and more than a hundred trillion times by 50, and 99 steps back it is exactly zero, too small for a float to hold. Figure 18.3 plots every step.

> 🖼️ **Figure 18.3:** The gradient that reaches each step of a length 100 sequence, before any training. Both cells lose it exponentially; the LSTM loses it more slowly.

The second row is the cell that was invented for this problem. In 1997 Sepp Hochreiter and Jürgen Schmidhuber published the **long short-term memory**, or **LSTM**. Besides the hidden state it keeps a separate memory, the **cell state**, that runs along the chain with very little arithmetic in its way, and three learned **gates**, each a sigmoid between 0 and 1, decide what to write into it, what to read out, and, in a version added by Felix Gers, Schmidhuber and Fred Cummins in 2000, what to forget. When the forget gate sits near 1, the memory passes from step to step almost untouched, and so does the gradient.

The measurement shows the LSTM fading more slowly, but fading all the same. The reason is in that last sentence: the gate helps only when it sits near 1, and a freshly built LSTM's forget gate starts near the middle. Gers and his colleagues advised starting it open. In 2015 Rafal Jozefowicz, Wojciech Zaremba and Ilya Sutskever, the same Sutskever as Chapters 1 and 15, tested thousands of recurrent designs and found that this one forgotten detail closed most of the gap between the LSTM and its best rivals. Test all three on the long memory task with sequences of length 50.

In [ ]:
def open_forget_gate(model):
    size = model.rnn.hidden_size
    for name, p in model.rnn.named_parameters():
        if "bias" in name:
            with torch.no_grad():
                p[size:2 * size].fill_(1.0)
    return model

for seed in [0, 1, 2]:
    results = []
    for build in [lambda: Reader(nn.RNN), lambda: Reader(nn.LSTM),
                  lambda: open_forget_gate(Reader(nn.LSTM))]:
        torch.manual_seed(seed)
        results.append(run(build(), make_first, 50, epochs=30, seed=seed))
    print(seed, results)

The three columns are the plain RNN, the LSTM as PyTorch builds it, and the LSTM with its forget gate started open. On all three seeds the first two sit at a coin toss and the third is perfect.

PyTorch stores the LSTM's four gates one after another in each bias vector, in the order input, forget, cell, output, so the forget gate's entries are the second quarter, `size:2 * size`. It also keeps two bias vectors and adds them, so setting both to 1 starts the gate with a bias of 2, which is inside the range of 1 to 2 that is usually recommended.

That result is the whole of Chapter 16 again in a new setting. Two networks with identical architecture and identical data, and whether one of them learns at all is decided by where a few of its numbers start.

## Where Sequences Go Next

For about twenty years the LSTM and a simpler relative called the GRU were how neural networks read sequences. They translated languages, recognised speech and wrote captions for photographs. But the chain itself remained a limit in two ways. Information from far back still has to survive being passed along step after step, as you just measured. And a chain has to be computed in order, one step after the last, so it cannot use the thousands of parallel units of a modern graphics card as fully as a convolutional network can.

In 2017 a team at Google published a paper with a deliberately provocative title, *Attention Is All You Need*. Their model, the **transformer**, dropped the chain entirely. Instead of passing a message from step to step, every position looks directly at every other position and decides how much attention to pay to each, which makes a word fifty places back exactly as reachable as the word next door, and lets every position be computed at once. Every large language model in use today is built on that idea.

Table 18.1 places the three ideas side by side.

**Table 18.1: Three ways to read a sequence. Each one removed an obstacle the one before it could not get past.**

| Model | How it carries the past | Where it lives |
|---|---|---|
| Recurrent network, 1990 | A hidden state passed step to step | This chapter |
| LSTM, 1997 and 2000 | A gated memory along the chain | This chapter |
| Transformer, 2017 | Every position attends to every other | Book 8 |

What carries over to Book 8 from here is more than history. Embeddings are the first layer of every language model. The hidden state you wrote by hand is the ancestor of the representations a transformer builds. And the habit of measuring the gradient before blaming the data will serve you there exactly as it did here.

## Practice and Review

Reading about deep learning is only half of learning it. Pulling an idea back out of your memory makes it stick much better than reading it again, so this section gives you two kinds of practice, one for your hands and one for your memory.

### Try It Yourself

1. **Give the bag a fair chance.** Change `make_order` so the label is whether a 1 appears at all, with half the sequences missing it, and confirm that the order blind model now does well.
2. **Find the RNN's limit.** Run the plain RNN on `make_first` at lengths 10, 20, 30 and 40 and find where it stops learning.
3. **Open it further.** Fill the forget gate bias with 2.0 instead of 1.0 and see whether the LSTM still learns at length 100.
4. **Try the GRU.** Pass `nn.GRU` to `Reader` with no special start and compare it with the plain LSTM on the long memory task.

### Check Your Understanding

Try to answer each question from memory before you look at the answers that follow.

1. **Why can a model that averages its embeddings never solve the order task?**
2. **What is an embedding, and what is it equivalent to?**
3. **How does one set of recurrent weights handle sequences of different lengths?**
4. **Why do gradients vanish across time in a recurrent network?**
5. **What does an LSTM's forget gate do, and why does its starting value matter?**
6. **What two limits of the recurrent chain did the transformer remove?**

### Answers

1. Because averaging throws away order, and every sequence in both classes contains the same symbols, so the two classes look identical to it.
2. A table with one learned row per symbol, looked up by the symbol's number. It is equivalent to multiplying a one hot vector by that table.
3. By applying the same cell, with the same weights, once per step, carrying a hidden state from each step to the next.
4. Because backpropagating through the steps multiplies by one factor per step, exactly like a deep network, and a long chain of factors below one shrinks the signal exponentially.
5. It decides how much of the memory to keep at each step. Started near the middle, it lets the memory fade like a plain RNN; started open, it lets memory and gradient pass along the chain, which here made the difference between a coin toss and a perfect score.
6. Information from far back no longer has to survive a long chain, because every position attends to every other directly, and the steps no longer have to be computed one after another.

## What Comes Next

You can now explain why order needs its own kind of model, turn symbols into vectors with an embedding, write a recurrent step by hand and check it against PyTorch, measure the vanishing gradient through time, and make an LSTM remember by starting its forget gate open. That completes Part 5.

- **Chapter 19, Saving, Loading and Running on a GPU,** turns from new kinds of network to the practical work of keeping a trained one and running it fast.
- **Book 8,** picks up embeddings and attention and builds them into language models.

Chapter 19 begins with an uncomfortable fact: every network you have trained in this book vanished the moment your Python session ended.